In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import jax

from popsim import param_utils
from popsim.modules.tearing import Island, Tearing, generate_disruption_phase_trajectory, generate_island_rotation_phase_trajectory
from popsim.simulate import simulate

dt = 1e-4 / 3  # s
time_base = param_utils.make_time_base(t0=0.0, t1=7.0, dt=dt)
config = Tearing.Config(
    magx_time=time_base
)


islands = [Island(2, 1), Island(3, 2)]

W = {island: 0.0 for island in islands}
F = {island: 0.0 for island in islands}
mode_phase={island: 0.0 for island in islands}

initial_state = Tearing.State(W=W, F=F, mode_phase=mode_phase)

rot_dur = 1.0
trigger_time = 5.0
disrupt_time = 6.5
dur_tq_to_spike = 1e-3
dur_cq = 10e-3
survival_time = 0.3
locking_dur = 0.2

params = Tearing.Params(
    rot_dur=1.0,  # s
    locking_dur=locking_dur,  # s
    disruption_phase=generate_disruption_phase_trajectory(
        disrupt_time, dur_tq_to_spike, time_base, dt
    ),
    island_rotation_phase=generate_island_rotation_phase_trajectory(
        trigger_time, rot_dur, locking_dur, time_base, dt
    ),
)

In [ ]:
jax.config.update("jax_platforms", "cpu")
tearing_module = Tearing(config=config, islands=islands)

sol_xarray = simulate(tearing_module, time_base, initial_state, params)

In [ ]:
from popsim.visualize import visualize_time_series

#sol_xarray.to_netcdf("tearing_simulation.nc")
visualize_time_series(sol_xarray, max_cols=2)